<a href="https://colab.research.google.com/github/emanfatimaa05/urdu-ocr-codesaviours-si26-eman/blob/main/week3_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/emanfatimaa05/urdu-ocr-codesaviours-si26-eman.git
%cd urdu-ocr-codesaviours-si26-eman

Cloning into 'urdu-ocr-codesaviours-si26-eman'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (389/389), done.
remote: Total 400 (delta 40), reused 218 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (400/400), 4.64 MiB | 32.12 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/urdu-ocr-codesaviours-si26-eman


In [3]:
!pip install transformers torch pillow pandas sentencepiece -q

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

In [6]:
class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [8]:
from transformers import TrOCRProcessor, ViTImageProcessor, RobertaTokenizer

image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

print('Processor loaded successfully!')

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Processor loaded successfully!


In [14]:
dataset = UrduOCRDataset('data/labels.csv', processor)
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

Dataset loaded: 205 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!


In [13]:
%cd /content
!rm -rf urdu-ocr-codesaviours-si26-eman
!git clone https://github.com/emanfatimaa05/urdu-ocr-codesaviours-si26-eman.git
%cd urdu-ocr-codesaviours-si26-eman

/content
Cloning into 'urdu-ocr-codesaviours-si26-eman'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (403/403), done.
remote: Compressing objects: 100% (392/392), done.
remote: Total 403 (delta 41), reused 218 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (403/403), 4.65 MiB | 48.13 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/urdu-ocr-codesaviours-si26-eman


In [15]:
torch.manual_seed(42)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Training samples: 164
Testing samples: 41
